## Import Packages

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import plotly

from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics import (
    roc_auc_score, recall_score, accuracy_score, f1_score, precision_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

print("All core packages imported successfully.")

In [ ]:
from sklearn import set_config

from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

import optuna
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")
set_config(transform_output="pandas")

In [ ]:
try:
    import dagshub
    dagshub.init(repo_owner="Ankitkumar1141", repo_name="Churn-modelling", mlflow=True)
    print("DagsHub initialized successfully.")
except Exception as e:
    print(f"[DagsHub Notice] {e}")

In [ ]:
try:
    import mlflow
    mlflow.set_tracking_uri("https://dagshub.com/Ankitkumar1141/Churn-modelling.mlflow")
    mlflow.set_experiment("Experiment 5 - XGB HP Tuning")
    print("MLflow tracking active on DagsHub.")
except Exception as e:
    print(f"[MLflow Notice] {e}")

In [ ]:
df = pd.read_csv("preprocessed.csv")
print("Loaded dataset shape:", df.shape)
df.head(3)

## Exploratory Checks

In [ ]:
df.dtypes

In [ ]:
df.isna().sum().sort_values(ascending=False).head(5)

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print(df["churned"].value_counts())
churn_rate = df["churned"].mean()
print("\nChurn rate: {:.2%}".format(churn_rate))

## Drop Unnecessary Columns

In [ ]:
drop_columns = [
    "has_balance",
    "products_per_tenure",
    "has_credit_card",
    "geo_france", "geo_uk", "geo_usa", "geo_spain", "geo_germany",
    "tenure", "salary",
    "customer_id",
    "account_id", "name", "location_id",
    "age_when_joined", "age",
    "churn_label",
    "has_credit_card_label",
    "is_active_label",
]

df = df.drop(columns=drop_columns)
print("Shape after dropping columns:", df.shape)
df.head(3)

In [ ]:
missing_cols = df.isna().any(axis=0).loc[lambda x: x].index.tolist()
print("Columns with missing values:", missing_cols)

In [ ]:
pct_missing_rows = df.isna().any(axis=1).mean().round(4) * 100
n_missing = df.isna().any(axis=1).sum()
print(f"Rows with missing values: {pct_missing_rows:.2f}%  ({n_missing:,} rows)")

## Train / Test Split

In [ ]:
X = df.drop(columns="churned")
y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Train size : {:,}  |  churn rate: {:.2%}".format(X_train.shape[0], y_train.mean()))
print("Test  size : {:,}  |  churn rate: {:.2%}".format(X_test.shape[0], y_test.mean()))

In [ ]:
print("Train shape:", X_train.shape)
print("Test  shape:", X_test.shape)
missing_in_train = X_train.isna().sum()
print("Missing in train (balance only):\n", missing_in_train[missing_in_train > 0])

## Column Type Identification

In [ ]:
numeric_cols     = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
binary_cols      = [c for c in numeric_cols if X[c].nunique() == 2]
scale_cols       = [c for c in numeric_cols if c not in binary_cols]

print("Numeric (to scale) :", scale_cols)
print("Binary pass-thru   :", binary_cols)
print("Categorical OHE    :", categorical_cols)

## Preprocessing Pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("scale",          MinMaxScaler(), scale_cols),
        ("nominal_encode", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), categorical_cols),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
)
preprocessor.set_output(transform="pandas")

knn_imputer = KNNImputer(n_neighbors=10)

In [ ]:
# fit on train, transform test
base_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("knn_imputer",  knn_imputer),
])

X_train_trans = base_pipeline.fit_transform(X_train)
X_test_trans  = base_pipeline.transform(X_test)

print("X_train transformed shape:", X_train_trans.shape)
print("X_test  transformed shape:", X_test_trans.shape)

In [ ]:
sampler = RandomUnderSampler(random_state=42)

# Apply resampling to training data only (never to test set)
X_train_resampled, y_train_resampled = sampler.fit_resample(X_train_trans, y_train)

In [ ]:
print("Before resampling — Train shape:", X_train_trans.shape)
print("Before resampling — Class distribution:")
print(y_train.value_counts())
print()
print("After  resampling — Train shape:", X_train_resampled.shape)
print("After  resampling — Class distribution:")
print(y_train_resampled.value_counts())

## XGBoost Hyperparameter Tuning with Optuna

Dedicated deep search across the **full XGBoost hyperparameter space**, building directly on the spot-check winner from Experiment 4.  
Each trial is logged as a nested MLflow run under **Experiment 5 — XGB HP Tuning**.  

**Primary metric: ROC-AUC** (maximised via 5-fold stratified CV on resampled training data).  

| Hyperparameter | Search Space | Description |
|---|---|---|
| `n_estimators` | 50 – 600 | Number of boosting rounds |
| `max_depth` | 2 – 12 | Max tree depth — controls complexity |
| `learning_rate` | 0.005 – 0.3 | Step-size shrinkage (η) |
| `subsample` | 0.5 – 1.0 | Row-sampling fraction per tree |
| `colsample_bytree` | 0.5 – 1.0 | Feature-sampling fraction per tree |
| `min_child_weight` | 1 – 10 | Min sum of instance weight in a leaf |
| `gamma` | 0.0 – 5.0 | Min loss reduction to make a split |
| `reg_alpha` | 1e-8 – 10.0 | L1 regularisation (log scale) |
| `reg_lambda` | 1e-8 – 10.0 | L2 regularisation (log scale) |

In [ ]:
def objective(trial):
    with mlflow.start_run(nested=True):
        params = {
            "n_estimators"     : trial.suggest_int("n_estimators", 50, 600),
            "max_depth"        : trial.suggest_int("max_depth", 2, 12),
            "learning_rate"    : trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
            "subsample"        : trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_weight" : trial.suggest_int("min_child_weight", 1, 10),
            "gamma"            : trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha"        : trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda"       : trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "random_state"     : 42,
            "n_jobs"           : -1,
            "verbosity"        : 0,
            "eval_metric"      : "logloss",
        }

        # log all hyperparameters
        mlflow.log_params(params)

        # build the model
        xgb = XGBClassifier(**params)

        # 5-fold stratified cross-validation on resampled training data
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        cv_scores = cross_validate(
            xgb,
            X_train_resampled,
            y_train_resampled,
            cv=cv,
            scoring={
                "roc_auc" : "roc_auc",
                "recall"  : "recall",
                "f1"      : "f1",
            },
            n_jobs=-1,
        )

        mean_roc_auc = cv_scores["test_roc_auc"].mean()
        mean_recall  = cv_scores["test_recall"].mean()
        mean_f1      = cv_scores["test_f1"].mean()

        # log CV metrics
        mlflow.log_metric("cv_mean_roc_auc", mean_roc_auc)
        mlflow.log_metric("cv_mean_recall",  mean_recall)
        mlflow.log_metric("cv_mean_f1",      mean_f1)

        return mean_roc_auc

In [ ]:
# create optuna study
study = optuna.create_study(direction="maximize", study_name="xgb_hp_tuning")

with mlflow.start_run(run_name="XGB HP Tuning — Best Model") as parent:
    # optimize the objective function
    study.optimize(objective, n_trials=50, n_jobs=-1, show_progress_bar=True)

    # log the best parameters
    mlflow.log_params(study.best_params)

    # log the best score
    mlflow.log_metric("best_cv_roc_auc", study.best_value)

In [ ]:
# best CV ROC-AUC achieved
print("Best CV ROC-AUC :", round(study.best_value, 4))

In [ ]:
# best parameters
print("Best Parameters:")
for k, v in study.best_params.items():
    print(f"  {k:20s}: {v}")

In [ ]:
# dataframe of all trials
study.trials_dataframe().sort_values("value", ascending=False)

## Optuna Visualisations

In [ ]:
# optimization history plot
optuna.visualization.plot_optimization_history(study)

In [ ]:
# hyperparameter importance plot
optuna.visualization.plot_param_importances(study)

In [ ]:
# slice plot -- individual parameter effect on objective
optuna.visualization.plot_slice(study)

In [ ]:
# parallel coordinate plot -- interactions between key params
optuna.visualization.plot_parallel_coordinate(
    study,
    params=["n_estimators", "max_depth", "learning_rate", "subsample", "colsample_bytree"]
)

## Train Best Model & Evaluate on Test Set

In [ ]:
# build best XGBoost model with tuned parameters
best_params = study.best_params.copy()
best_params.update({
    "random_state" : 42,
    "n_jobs"       : -1,
    "verbosity"    : 0,
    "eval_metric"  : "logloss",
})

best_xgb = XGBClassifier(**best_params)

# train on resampled data
best_xgb.fit(X_train_resampled, y_train_resampled)
print("Best XGBoost model trained.")

In [ ]:
# predictions on full train + test (pre-transformed, unresampled)
y_pred_train = best_xgb.predict(X_train_trans)
y_pred_test  = best_xgb.predict(X_test_trans)
y_prob_train = best_xgb.predict_proba(X_train_trans)[:, 1]
y_prob_test  = best_xgb.predict_proba(X_test_trans)[:, 1]

train_acc = accuracy_score(y_train, y_pred_train)
test_acc  = accuracy_score(y_test, y_pred_test)
train_rec = recall_score(y_train, y_pred_train)
test_rec  = recall_score(y_test, y_pred_test)
train_auc = roc_auc_score(y_train, y_prob_train)
test_auc  = roc_auc_score(y_test, y_prob_test)

print(f"Train accuracy : {train_acc:.4f}")
print(f"Test  accuracy : {test_acc:.4f}")
print(f"Train recall   : {train_rec:.4f}")
print(f"Test  recall   : {test_rec:.4f}")
print(f"Train ROC-AUC  : {train_auc:.4f}")
print(f"Test  ROC-AUC  : {test_auc:.4f}")

In [ ]:
print("\n=== Classification Report (Test Set) ===")
print(classification_report(y_test, y_pred_test, target_names=["Stayed", "Churned"]))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Stayed", "Churned"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — XGBoost (Tuned)", fontweight="bold")
plt.tight_layout()
plt.show()

## Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "roc_auc"  : "roc_auc",
    "recall"   : "recall",
    "precision": "precision",
    "f1"       : "f1",
    "accuracy" : "accuracy",
}

In [ ]:
# Full pipeline: preprocessor + KNN imputer + resampler + best XGB
cv_pipeline = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("knn_imputer",  KNNImputer(n_neighbors=10)),
    ("sampler",      RandomUnderSampler(random_state=42)),
    ("classifier",   best_xgb),
])

cv_results = cross_validate(
    cv_pipeline,
    X_train,       # raw X_train — pipeline handles everything inside each fold
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
)

results_df = pd.DataFrame(cv_results)
results_df

In [ ]:
print("=== CV Summary (5-Fold Stratified) ===")
print(f"Mean CV ROC-AUC  : {results_df['test_roc_auc'].mean():.4f} +/- {results_df['test_roc_auc'].std():.4f}")
print(f"Mean CV Recall   : {results_df['test_recall'].mean():.4f} +/- {results_df['test_recall'].std():.4f}")
print(f"Mean CV Precision: {results_df['test_precision'].mean():.4f} +/- {results_df['test_precision'].std():.4f}")
print(f"Mean CV F1       : {results_df['test_f1'].mean():.4f} +/- {results_df['test_f1'].std():.4f}")
print(f"Mean CV Accuracy : {results_df['test_accuracy'].mean():.4f} +/- {results_df['test_accuracy'].std():.4f}")

## Log Experiment to MLflow

In [ ]:
try:
    with mlflow.start_run(run_name="XGB HP Tuning — Final Evaluation"):
        mlflow.log_param("experiment_type",    "XGB Hyperparameter Tuning")
        mlflow.log_param("imbalance_strategy", "RandomUnderSampler")
        mlflow.log_param("imputer",            "KNNImputer(n_neighbors=10)")
        mlflow.log_param("n_optuna_trials",    50)
        mlflow.log_param("dataset_rows",       X.shape[0])
        mlflow.log_param("dataset_features",   X.shape[1])
        mlflow.log_params(best_params)

        mlflow.log_metric("training_accuracy", accuracy_score(y_train, y_pred_train))
        mlflow.log_metric("training_recall",   recall_score(y_train, y_pred_train))
        mlflow.log_metric("training_roc_auc",  roc_auc_score(y_train, y_prob_train))
        mlflow.log_metric("test_accuracy",     accuracy_score(y_test, y_pred_test))
        mlflow.log_metric("test_recall",       recall_score(y_test, y_pred_test))
        mlflow.log_metric("test_precision",    precision_score(y_test, y_pred_test))
        mlflow.log_metric("test_f1",           f1_score(y_test, y_pred_test))
        mlflow.log_metric("test_roc_auc",      roc_auc_score(y_test, y_prob_test))
        mlflow.log_metric("cv_mean_recall",    results_df['test_recall'].mean())
        mlflow.log_metric("cv_mean_roc_auc",   results_df['test_roc_auc'].mean())
        mlflow.log_metric("cv_mean_f1",        results_df['test_f1'].mean())
        mlflow.log_metric("optuna_best_cv_roc_auc", study.best_value)

        for i, s in enumerate(results_df['test_roc_auc']):
            mlflow.log_metric("cv_roc_auc_fold", s, step=i)
        for i, s in enumerate(results_df['test_recall']):
            mlflow.log_metric("cv_recall_fold", s, step=i)

        # log model artifact
        signature = mlflow.models.infer_signature(
            model_input=X_train_resampled.sample(20, random_state=42),
            model_output=best_xgb.predict_proba(X_train_resampled.sample(20, random_state=42))[:, 1],
        )
        mlflow.sklearn.log_model(best_xgb, artifact_path="xgb_tuned_model", signature=signature)

    print("Experiment logged to MLflow.")
except Exception as e:
    print(f"[MLflow Notice] {e}")

## Final Summary

In [ ]:
# Comparison: spot-check baseline (Exp 4) vs tuned (Exp 5)
summary = pd.DataFrame({
    "Metric": [
        "Test ROC-AUC", "Test Recall", "Test Precision", "Test F1", "Test Accuracy",
        "CV ROC-AUC (mean)", "CV Recall (mean)", "CV F1 (mean)", "CV Accuracy (mean)",
    ],
    "Exp 4: XGB Spot-Check (Baseline)": [
        0.8473, 0.7224, 0.4696, 0.5692, 0.7775,
        0.8368, 0.7178, 0.5607, None,
    ],
    "Exp 5: XGB HP Tuning (This Notebook)": [
        round(roc_auc_score(y_test, y_prob_test),           4),
        round(recall_score(y_test, y_pred_test),            4),
        round(precision_score(y_test, y_pred_test),         4),
        round(f1_score(y_test, y_pred_test),                4),
        round(accuracy_score(y_test, y_pred_test),          4),
        round(results_df['test_roc_auc'].mean(),            4),
        round(results_df['test_recall'].mean(),             4),
        round(results_df['test_f1'].mean(),                 4),
        round(results_df['test_accuracy'].mean(),           4),
    ]
})
summary

In [ ]:
print(f">>> Optuna best CV ROC-AUC : {study.best_value:.4f}")
print(f">>> Test  ROC-AUC (tuned)  : {roc_auc_score(y_test, y_prob_test):.4f}")
print(f">>> Test  Recall   (tuned) : {recall_score(y_test, y_pred_test):.4f}")
print(f">>> Test  F1       (tuned) : {f1_score(y_test, y_pred_test):.4f}")
print()
print(">>> Best Hyperparameters:")
for k, v in study.best_params.items():
    print(f"    {k:20s}: {v}")
print()
print(">>> Next: 07_Churn_LGBM_HP_Tuning.ipynb")